<a href="https://colab.research.google.com/github/majitomojito/ladybugs-mlops/blob/mlops/colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLOps Demo Project - Google Colab

A complete, reproducible Machine Learning Operations (MLOps) pipeline demonstrating best practices for training, evaluating, and deploying ML models.

## 1. Install Dependencies

In [ ]:
!pip install -q scikit-learn numpy pandas pyyaml joblib pytest pytest-cov

## 2. Clone the Repository

In [ ]:
import os

# Clone repository from mlops branch
!git clone -b mlops https://github.com/majitomojito/ladybugs-mlops.git /content/mlops-demo
os.chdir('/content/mlops-demo')

print("\nRepository cloned successfully!")


The ML pipeline executes:

1. **Load Data**: Iris dataset (150 samples, 4 features, 3 classes)
2. **Preprocessing**: Normalize features using StandardScaler
3. **Train Model**: Random Forest classifier
4. **Evaluate**: Calculate accuracy, precision, recall, F1-score
5. **Save Artifacts**: Store model, scaler, and metrics

## 3. Run Tests

In [ ]:
import subprocess
import sys

# Run pytest
result = subprocess.run([sys.executable, '-m', 'pytest', 'tests/', '-v'], 
                       capture_output=False)

if result.returncode == 0:
    print("\nAll tests passed!")
else:
    print("\nSome tests failed")

## 4. Run the ML Pipeline

In [ ]:
import logging
import sys
from src.pipeline import run_pipeline

# Configure logging to write directly to stdout for Jupyter/Colab compatibility
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    stream=sys.stdout,
    force=True  # Override any existing config
)

# Run the training pipeline—assign return value to suppress dict output
metrics = run_pipeline()

# Print summary
print("\n" + "=" * 70)
print("Pipeline completed successfully!")
print("=" * 70)

## 5. View Results

In [ ]:
import json
import os

# Load and display metrics
metrics_path = 'models/metrics/metrics.json'

if os.path.exists(metrics_path):
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    print("\n📈 Model Performance Summary:")
    print("=" * 50)
    for metric_name, metric_value in metrics.items():
        print(f"{metric_name.capitalize():12} : {metric_value:.4f}")
    print("=" * 50)
else:
    print("Metrics file not found. Run the pipeline first.")

## (Optional) Explore the Code and change the pipeline

### Data Loading Module

In [4]:
# Display data loader code - comment out below code
# with open('src/data/data_loader.py', 'r') as f:
#     print(f.read())

### Model Training Module

In [3]:
# Display model code - comment out below code
# with open('src/models/model.py', 'r') as f:
#     print(f.read())

### Customize the Pipeline

Edit the configuration file to experiment with different settings:

In [ ]:
# Display current configuration - comment out below code
# with open('config/config.yaml', 'r') as f:
#     print(f.read())

### Try Different Models

In [ ]:
# Example: Try Gradient Boosting instead of Random Forest - comment out below code
# from sklearn.ensemble import GradientBoostingClassifier
# from src.data import load_data, preprocess_data
# from src.models import evaluate_model, save_metrics
# from pathlib import Path

# # Load and preprocess data
# X_train, X_test, y_train, y_test = load_data()
# X_train_scaled, X_test_scaled, _ = preprocess_data(X_train, X_test)

# # Train alternative model
# print("Training Gradient Boosting model...")
# gb_model = GradientBoostingClassifier(n_estimators=100, max_depth=5)
# gb_model.fit(X_train_scaled, y_train)

# # Evaluate
# metrics = evaluate_model(gb_model, X_test_scaled, y_test)

# print("\nGradient Boosting Results:")
# print("=" * 50)
# for metric_name, metric_value in metrics.items():
#     print(f"{metric_name.capitalize():12} : {metric_value:.4f}")
# print("=" * 50)

## 6. Visualize Data Distribution

Let's visualize the original dataset to understand what the model learned from:

- The first figure (4 panel histogram) shows training-class feature distributions.
- Each feature is plotted per iris species so you can see the separation vs overlap.
- This sets the reference for what “in-distribution” looks like.

In the next section (Data Drift), we compare these plots to production data:
- Drifted data histogram overlays show exactly how the new production distribution shifts.
- A large shift or extra overlap indicates the model will likely fail out-of-sample.
- This makes the later performance drop numbers easier to interpret and justify.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris

# Load original data
iris = load_iris()
X = iris.data
y = iris.target

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Original Training Data Distribution\n(Feature Distributions by Class)', fontsize=14, fontweight='bold')

feature_names = iris.feature_names
colors = ['red', 'blue', 'green']
class_names = iris.target_names

for idx, (ax, feature_idx) in enumerate(zip(axes.flat, range(4))):
    for class_idx in range(3):
        mask = y == class_idx
        ax.hist(X[mask, feature_idx], alpha=0.6, label=class_names[class_idx], color=colors[class_idx], bins=15)
    ax.set_xlabel(feature_names[feature_idx])
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Original data shows balanced distribution across all 3 iris species")

You can see three classes and for each feature we have a characteristic **shape** and **separation**. This is the baseline the model learned from.
So this is the world where our model performs well: data distributions are familiar, and class representation is stable.

## 7. Simulate Data Drift

**What is Data Drift?**
Data drift occurs when the distribution of production data differs significantly from training data. This is a **critical MLOps challenge** because:

- Models trained on old data may fail on new data
- Metrics worsen
- Model seems fine but results in poor quality predictions
- These can lead to wrong decisions

Let's create a **drifted dataset** to show this problem:

In [ ]:
import numpy as np
from typing import Iterable, Optional, Sequence, Tuple, Union

def simulate_data_drift(
    n_total: int = 450,
    *,
    # Cluster specs: list of dicts with keys: "weight" OR "n", "loc", "scale"
    clusters: Optional[Sequence[dict]] = None,
    n_features: int = 4,
    seed: Optional[int] = None,
    # Label generation (concept drift / prior shift)
    label_probs: Sequence[float] = (0.55, 0.35, 0.10),
    labels: Sequence[int] = (0, 1, 2),
    # Optional: keep some relationship between X and y (0.0 = fully random, 1.0 = fully cluster-driven)
    x_y_coupling: float = 0.0,
    # Shuffle output
    shuffle: bool = True,
    # Return also which cluster each row came from (useful for debugging/plots)
    return_cluster_ids: bool = False,
) -> Union[Tuple[np.ndarray, np.ndarray], Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    """
    Simulate drifted 'production' data with controllable feature drift + label drift.

    Parameters
    ----------
    n_total:
        Total number of samples to generate (ignored if clusters specify explicit 'n' and sum to a different total).
    clusters:
        Cluster definitions. If None, uses the cluster setup from your snippet.
        Each cluster dict supports:
          - Either {"weight": float, "loc": [...], "scale": [...]}  (weights normalized; uses n_total)
            OR {"n": int, "loc": [...], "scale": [...]}            (explicit counts)
          - loc/scale can be length n_features or scalars.
    n_features:
        Number of features (columns).
    seed:
        RNG seed for reproducibility.
    label_probs / labels:
        Defines y distribution when y is random (x_y_coupling=0) and as a fallback mixture when coupling > 0.
    x_y_coupling:
        Blend between random labels and cluster-based labels.
        - 0.0 => labels are drawn independently of X (pure concept/prior drift simulation)
        - 1.0 => labels are determined by cluster membership (more like covariate drift only)
        Values in between => mix of both.
    shuffle:
        Shuffle rows to remove ordering by cluster.
    return_cluster_ids:
        If True, also returns an array with the originating cluster index per row.

    Returns
    -------
    X, y  (and optionally cluster_ids)
    """
    rng = np.random.default_rng(seed)

    # Default clusters: mirrors your code (300 + 90 + 60 = 450, 4 features)
    if clusters is None:
        clusters = [
            dict(weight=300 / 450, loc=[6.0, 3.0, 4.5, 1.3], scale=[1.0, 0.8, 1.0, 0.6]),
            dict(weight=90 / 450,  loc=[5.2, 3.3, 1.6, 0.4], scale=[0.4, 0.3, 0.3, 0.2]),
            dict(weight=60 / 450,  loc=[6.8, 2.9, 5.8, 1.8], scale=[0.5, 0.4, 0.5, 0.3]),
        ]

    # Determine per-cluster counts
    if all("n" in c for c in clusters):
        n_per = [int(c["n"]) for c in clusters]
        n_total_eff = int(sum(n_per))
    else:
        weights = np.array([c.get("weight", 0.0) for c in clusters], dtype=float)
        if weights.sum() <= 0:
            raise ValueError("clusters must provide either explicit 'n' for all clusters or positive 'weight' values.")
        weights = weights / weights.sum()

        # Multinomial ensures counts sum exactly to n_total
        n_per = list(rng.multinomial(n_total, weights))
        n_total_eff = n_total

    # Build X
    X_parts = []
    cluster_ids_parts = []
    for i, (c, n_i) in enumerate(zip(clusters, n_per)):
        loc = np.asarray(c["loc"], dtype=float)
        scale = np.asarray(c["scale"], dtype=float)

        if loc.size == 1:
            loc = np.full(n_features, float(loc))
        if scale.size == 1:
            scale = np.full(n_features, float(scale))

        if loc.size != n_features or scale.size != n_features:
            raise ValueError(f"Cluster {i}: loc/scale must be scalar or length n_features={n_features}.")

        X_i = rng.normal(loc=loc, scale=scale, size=(n_i, n_features))
        X_parts.append(X_i)
        cluster_ids_parts.append(np.full(n_i, i, dtype=int))

    X = np.vstack(X_parts) if X_parts else np.empty((0, n_features), dtype=float)
    cluster_ids = np.concatenate(cluster_ids_parts) if cluster_ids_parts else np.empty((0,), dtype=int)

    # Generate y
    labels = np.asarray(labels, dtype=int)
    label_probs = np.asarray(label_probs, dtype=float)
    label_probs = label_probs / label_probs.sum()

    # Cluster-driven labels (simple mapping: cluster 0->labels[0], cluster 1->labels[0], cluster 2->labels[2] by default)
    # You can override by providing "cluster_label" in a cluster dict.
    cluster_label_map = []
    for i, c in enumerate(clusters):
        if "cluster_label" in c:
            cluster_label_map.append(int(c["cluster_label"]))
        else:
            # Default: map each cluster to a label (reasonable for Iris-like demo)
            # middle ambiguous cluster -> 0, setosa-like -> 0, virginica-like -> 2
            default_map = {0: labels[0], 1: labels[0], 2: labels[-1]}
            cluster_label_map.append(default_map.get(i, labels[0]))

    y_cluster = np.array([cluster_label_map[i] for i in cluster_ids], dtype=int)
    y_random = rng.choice(labels, size=n_total_eff, p=label_probs)

    x_y_coupling = float(np.clip(x_y_coupling, 0.0, 1.0))
    if x_y_coupling == 0.0:
        y = y_random
    elif x_y_coupling == 1.0:
        y = y_cluster
    else:
        # Per-row mix: choose cluster label with probability coupling, else random
        choose_cluster = rng.random(n_total_eff) < x_y_coupling
        y = np.where(choose_cluster, y_cluster, y_random)

    # Shuffle
    if shuffle:
        idx = rng.permutation(n_total_eff)
        X = X[idx]
        y = y[idx]
        cluster_ids = cluster_ids[idx]

    if return_cluster_ids:
        return X, y, cluster_ids
    return X, y


In [ ]:
X_drifted, y_drifted = simulate_data_drift(seed=42)
# Try it out!
# X_drifted, y_drifted = simulate_data_drift(
#      n_total=600,
#      seed=7,
#      label_probs=(0.45, 0.40, 0.15),
#      x_y_coupling=0.5,  # 50% cluster-driven labels, 75% random
#  )

print("Drifted dataset created.")
print(f"Class distribution in original test set: {np.bincount(y_test)}")
print(f"Class distribution in drifted data: {np.bincount(y_drifted)}")
print(f"\nFeature means - Original test: {X_test.mean(axis=0)}")
print(f"Feature means - Drifted data: {X_drifted.mean(axis=0)}")

Notice the class balance we had in the original test set: [10, 10, 10]. That’s perfectly balanced. When we evaluate models offline on a balanced set, metrics often look clean and stable, because each class is represented equally and the feature distributions match what the model expects.

In the drifted dataset, the class balance changed significantly. We now have a lot more data and a different mix of cases. And we know that ML models can be sensitive to class imbalance.

### Visualize Data Drift

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Drifted Training Data Distribution\n(Feature Distributions by Class)', fontsize=14, fontweight='bold')

feature_names = iris.feature_names
colors = ['red', 'blue', 'green']
class_names = iris.target_names

for idx, (ax, feature_idx) in enumerate(zip(axes.flat, range(4))):
    for class_idx in range(3):
        mask = y_drifted == class_idx
        ax.hist(X_drifted[mask, feature_idx], alpha=0.6, label=class_names[class_idx], color=colors[class_idx], bins=15)
    ax.set_xlabel(feature_names[feature_idx])
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Within each class, the feature distributions have shifted too. Notice the peaks and spreads: some classes show broader ranges, some show shifted centers, the overlap between classes changes.
This is important because many models rely on the idea that “class A tends to have feature ranges like X, class B like Y.” If those class-conditional patterns change, then:
the decision boundary the model learned is no longer optimal,
you get more overlap between classes,
and the model becomes more uncertain and more error-prone.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Data Drift: Original Training vs Production Data\n(How the distribution changed in production)', 
             fontsize=14, fontweight='bold', color='red')

for idx, (ax, feature_idx) in enumerate(zip(axes.flat, range(4))):
    # Plot original test data
    ax.hist(X_test[:, feature_idx], alpha=0.6, label='Original (Training)', 
            color='blue', bins=20, density=True)
    
    # Plot drifted data
    ax.hist(X_drifted[:, feature_idx], alpha=0.6, label='Drifted (Production)', 
            color='red', bins=20, density=True)
    
    ax.set_xlabel(feature_names[feature_idx], fontsize=11)
    ax.set_ylabel('Density')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    ax.set_title(f'Feature {feature_idx}: {feature_names[feature_idx]}')

plt.tight_layout()
plt.show()

print("Data distribution has changed")
print("This will impact model performance in ways not captured by accuracy")

Some of these shifts may look small, but drift isn’t only about the mean changing, it can be changes in variance, tails, or relationships between features. In Ml, even subtle distribution shifts can move a lot of points closer to the model’s decision boundaries, which increases errors.

## 8. Evaluate Model on Drifted Data - Why MLOps Matters

Now let's train a model on the original data and test it on both original and drifted data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Train model on original data
X_train, X_test, y_train, y_test = load_data()
X_train_scaled, X_test_scaled, scaler = preprocess_data(X_train, X_test)

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate on ORIGINAL test data
y_pred_original = model.predict(X_test_scaled)
metrics_original = {
    "accuracy": accuracy_score(y_test, y_pred_original),
    "precision": precision_score(y_test, y_pred_original, average="weighted"),
    "recall": recall_score(y_test, y_pred_original, average="weighted"),
    "f1": f1_score(y_test, y_pred_original, average="weighted"),
}

# Evaluate on DRIFTED data (preprocessed with ORIGINAL scaler!!!)
# This is realistic - we use the production scaler from training
X_drifted_scaled = scaler.transform(X_drifted)
y_pred_drifted = model.predict(X_drifted_scaled)
metrics_drifted = {
    "accuracy": accuracy_score(y_drifted, y_pred_drifted),
    "precision": precision_score(y_drifted, y_pred_drifted, average="weighted"),
    "recall": recall_score(y_drifted, y_pred_drifted, average="weighted"),
    "f1": f1_score(y_drifted, y_pred_drifted, average="weighted"),
}

print("=" * 70)

print("=" * 70)
print("\nOriginal Test Data (Training Distribution):")
for metric, value in metrics_original.items():
    print(f"   {metric.capitalize():12} : {value:.4f}")

print("\nDRIFTED Production Data (Changed Distribution):")
for metric, value in metrics_drifted.items():
    print(f"   {metric.capitalize():12} : {value:.4f}")

print("\nPerformance Drop (%):")
for metric in metrics_original.keys():
    drop = (metrics_original[metric] - metrics_drifted[metric]) * 100
    symbol = "↓" if drop > 0 else "↑"
    print(f"   {metric.capitalize():12} : {symbol} {abs(drop):6.2f}%")

In [ ]:
# Create comparison visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
metrics_names = list(metrics_original.keys())
original_values = [metrics_original[m] for m in metrics_names]
drifted_values = [metrics_drifted[m] for m in metrics_names]

x = np.arange(len(metrics_names))
width = 0.35

bars1 = ax1.bar(x - width/2, original_values, width, label='Training Distribution', color='green', alpha=0.7)
bars2 = ax1.bar(x + width/2, drifted_values, width, label='Drifted Distribution', color='red', alpha=0.7)

ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Model Performance Drop Due to Data Drift', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([m.capitalize() for m in metrics_names])
ax1.legend(fontsize=11)
ax1.set_ylim([0, 1.1])
ax1.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=9)

# Performance degradation chart
performance_drop = [(metrics_original[m] - metrics_drifted[m]) * 100 for m in metrics_names]
colors_drop = ['red' if x > 0 else 'green' for x in performance_drop]
bars3 = ax2.bar(metrics_names, performance_drop, color=colors_drop, alpha=0.7)

ax2.set_ylabel('Performance Drop (%)', fontsize=12)
ax2.set_title('Performance Degradation (Percentage Points)', fontsize=13, fontweight='bold')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars3:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom' if height > 0 else 'top', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

**Without MLOps monitoring:**
- Model deployed and forgotten
- Data distribution changes over time (new users, new products, seasonal changes)
- Model performance degrades **silently**
- Business makes wrong decisions based on bad predictions
-  **Cost**: Revenue loss, customer churn, reputation damage

**Example from this notebook:**
- Our model had **90% accuracy** on training data
- **Same model** on drifted data: **~60% accuracy** (varies by class)
- This 30% drop can cause decisions to be made on incorrect predictions.

### MLOps Monitoring

1. **Monitor data distribution** - Detect when production data drifts from training
2. **Track model metrics** - Alert when accuracy drops below threshold
3. **Version models** - Know exactly what code/data produced each model
4. **Automate retraining** - Automatically retrain when performance degrades
5. **CI/CD for ML** - Test model changes before deployment
6. **Audit trail** - Track all model decisions for compliance

### MLOps Tools Solve This

- **Data Drift Monitoring**: Evidently, WhyLabs, Soda
- **Experiment Tracking**: MLflow, Weights & Biases, Neptune
- **CI/CD for ML**: GitHub Actions, Jenkins, GitLab CI
- **Model Serving**: Seldon, BentoML, TensorFlow Serving
- **Retraining Automation**: Kubeflow, Apache Airflow, Prefect

## 📉 Per-Class Performance Analysis

Even more critical: **different classes degrade differently!**
This is why MLOps requires **detailed metrics**, not just overall accuracy:

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Per-class performance
print("\n" + "="*70)
print("PER-CLASS ANALYSIS")
print("="*70)

print("\nORIGINAL DATA - Per-class performance:")
print(classification_report(y_test, y_pred_original, target_names=iris.target_names, digits=4))

print("\nDRIFTED DATA - Per-class performance:")
print(classification_report(y_drifted, y_pred_drifted, target_names=iris.target_names, digits=4))

# Calculate per-class degradation
from sklearn.metrics import precision_recall_fscore_support

prec_orig, rec_orig, f1_orig, _ = precision_recall_fscore_support(y_test, y_pred_original)
prec_drift, rec_drift, f1_drift, _ = precision_recall_fscore_support(y_drifted, y_pred_drifted)

print("\nPRECISION DROP PER CLASS:")
for i, class_name in enumerate(iris.target_names):
    drop = (prec_orig[i] - prec_drift[i]) * 100
    print(f"   {class_name:12} : {drop:6.2f}% drop")

# Visualize confusion matrices
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Original confusion matrix
cm_original = confusion_matrix(y_test, y_pred_original)
im1 = ax1.imshow(cm_original, cmap='Blues', aspect='auto')
ax1.set_title('Confusion Matrix: Original Data\n(Model Trained On)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
ax1.set_xticks([0, 1, 2])
ax1.set_yticks([0, 1, 2])
ax1.set_xticklabels(iris.target_names)
ax1.set_yticklabels(iris.target_names)
for i in range(3):
    for j in range(3):
        text = ax1.text(j, i, cm_original[i, j],
                       ha="center", va="center", color="black", fontweight='bold')
plt.colorbar(im1, ax=ax1)

# Drifted confusion matrix
cm_drifted = confusion_matrix(y_drifted, y_pred_drifted)
im2 = ax2.imshow(cm_drifted, cmap='Reds', aspect='auto')
ax2.set_title('Confusion Matrix: Drifted Data\n(Model Fails Here)', fontsize=12, fontweight='bold', color='red')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
ax2.set_xticks([0, 1, 2])
ax2.set_yticks([0, 1, 2])
ax2.set_xticklabels(iris.target_names)
ax2.set_yticklabels(iris.target_names)
for i in range(3):
    for j in range(3):
        text = ax2.text(j, i, cm_drifted[i, j],
                       ha="center", va="center", color="black", fontweight='bold')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()



### Resources
- 📚 [Scikit-learn Documentation](https://scikit-learn.org/)
- 🧪 [Pytest Documentation](https://docs.pytest.org/)
- 🔄 [MLflow Documentation](https://mlflow.org/)
- 📊 [GitHub Actions Documentation](https://docs.github.com/en/actions)
